[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session1/BMEN35_Ex1_02_Decision_tree_step_by_step.ipynb)

# Decision trees, one step at a time

In this notebook we will work through a very small decision-tree example. We will first look at every observation, then try one split by hand, calculate entropy, and compare possible splits. After that we will repeat the process on one branch of the tree.

This example is adapted from the decision-tree example used in the course. There are only ten observations with two features, `x1` and `x2`. The values are made up; they are not patient measurements. The small size is useful because we can see exactly what each line of code does.

Run one cell at a time. Before each calculation, try to guess what the output will look like. The code is written out step by step on purpose.

## 1. Import the libraries

NumPy holds the feature values and labels. Matplotlib draws the observations. We will use `entropy` from SciPy after we have looked at the labels in the child groups.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

## 2. Enter the data

Each row of `X` is one observation. Column 0 is `x1`, and column 1 is `x2`. The matching value in `y` is the class label for that row.

For the plot we will use red for class 0 and blue for class 1, as in the example from the book. The row numbers are added to the plot so we can find each point in the data.

In [ ]:
# Two feature values (x1, x2) for each of the ten observations.
X = np.array([
    [9, 2],
    [1, 4],
    [4, 6],
    [4, 1],
    [1, 2],
    [1, 8],
    [6, 4],
    [7, 9],
    [9, 8],
    [9, 6]
])

# One class label for each row in X. The row order must match.
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

colors = np.array(['red', 'blue'])

print('Number of observations:', len(y))
print('Number of features:', X.shape[1])
print('Class labels:', np.unique(y))

## 3. Always look at the data first

Before we start calculating splits, let's make a table and a scatter plot. The table shows the actual coordinates and labels. The plot puts `x1` on the horizontal axis and `x2` on the vertical axis; color tells us the class.

This is a tiny dataset, so we can plot every point. In a real dataset we would also need to know what the features measure and how the labels were collected.

In [ ]:
# Put the two feature columns and the class label into one table.
data_table = np.column_stack((X, y))
print('Columns are x1, x2 and class label:')
print(data_table)

# Plot one point per row. The class number selects the point color.
plt.scatter(X[:, 0], X[:, 1], c=colors[y], s=90, edgecolor='black')

# Write the row number beside each point so we can find it in the table.
for row_number in range(len(X)):
    x1_value = X[row_number, 0]
    x2_value = X[row_number, 1]
    plt.annotate(
        str(row_number),
        (x1_value, x2_value),
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('The ten observations')
plt.grid(alpha=0.25)
plt.show()

### Stop and look at the plot

1. Which color is mostly on the left side of the plot?
2. Which points look like they might be difficult to separate from the other class?
3. Can you think of a simple rule using only `x1` or `x2` that would separate some of the points?

A decision tree makes rules like this. Let's first create the possible places where a rule could split the data.

## 4. Find candidate split points

A split is a rule such as `x1 < 5`. If two observations have `x1` values 4 and 6, a useful split can be placed halfway between them, at 5. That way identical feature values will not be sent to different sides.

`np.unique` gives us the distinct values in a column. The expression below adds neighboring values and divides by two. We will do this separately for `x1` and `x2`, so it is clear which feature each threshold belongs to.

In [ ]:
# The first column of X contains all x1 values.
x1_values = np.unique(X[:, 0])
print('Unique x1 values:', x1_values)

# Put each possible x1 threshold halfway between neighboring values.
split_x1 = (x1_values[1:] + x1_values[:-1]) / 2
print('Possible x1 split points:', split_x1)

In [ ]:
# Now do the same thing for the second feature, x2.
x2_values = np.unique(X[:, 1])
print('Unique x2 values:', x2_values)

split_x2 = (x2_values[1:] + x2_values[:-1]) / 2
print('Possible x2 split points:', split_x2)

## 5. Try one split: `x2 < 3`

Let's start with one possible rule: send a point left if `x2 < 3`, and send all other points right. A Boolean mask gives us `True` for the rows that satisfy the rule and `False` for the others.

We will print the masks and the labels on both sides. This makes the split concrete before we try to score it.

In [ ]:
# True means that the observation satisfies the rule x2 < 3.
left_side = X[:, 1] < 3
right_side = ~left_side

print('Left-side mask:', left_side)
print('Labels sent left:', y[left_side])
print('Labels sent right:', y[right_side])
print('Number sent left:', left_side.sum())
print('Number sent right:', right_side.sum())

The left side contains three observations, and all three are class 0. The right side contains the other seven observations and is mixed. The split has made one pure group, but the other group still contains both classes.

## 6. Count the classes in each group

Entropy is calculated from class proportions. First we count how many class 0 and class 1 labels are on each side. `np.sum(labels == 0)` counts the class 0 labels; the same expression works for class 1.

Let's do the left and right groups separately and keep the counts visible.

In [ ]:
left_labels = y[left_side]
right_labels = y[right_side]

left_class_0 = np.sum(left_labels == 0)
left_class_1 = np.sum(left_labels == 1)
right_class_0 = np.sum(right_labels == 0)
right_class_1 = np.sum(right_labels == 1)

print('Left group: class 0 =', left_class_0, ', class 1 =', left_class_1)
print('Right group: class 0 =', right_class_0, ', class 1 =', right_class_1)

## 7. Calculate entropy for each group

Entropy is a measure of how mixed the labels are. For two classes, if the class proportions are $p_0$ and $p_1$, then

$$
H = -p_0\log_2(p_0)-p_1\log_2(p_1).
$$

If a group contains only one class, its entropy is 0. If it contains half of each class, its entropy is 1 bit. SciPy's `entropy` function expects proportions, so we divide the counts by the number of observations in each group.

In [ ]:
# Convert the class counts on the left into proportions.
left_counts = np.array([left_class_0, left_class_1])
left_proportions = left_counts / left_counts.sum()
left_entropy = entropy(left_proportions[left_proportions > 0], base=2)

# Do the same for the right group.
right_counts = np.array([right_class_0, right_class_1])
right_proportions = right_counts / right_counts.sum()
right_entropy = entropy(right_proportions[right_proportions > 0], base=2)

print('Left proportions:', left_proportions)
print('Left entropy:', round(left_entropy, 3))
print('Right proportions:', right_proportions)
print('Right entropy:', round(right_entropy, 3))

The left group is pure, so its entropy is zero. The right group is mixed, so its entropy is greater than zero. A split should not be judged only by the entropy on one side: we need to combine both sides, and a large group should count more than a small group.

## 8. Calculate the weighted entropy

The split score is the weighted average of the two child entropies. The weight is the fraction of all observations that went into that child:

$$
H_{split} = \frac{n_L}{n}H_L + \frac{n_R}{n}H_R.
$$

Let's calculate the weights separately, then calculate the score. The parent entropy is the entropy before splitting; comparing it with the split score tells us how much the split reduced the mixing.

In [ ]:
number_left = len(left_labels)
number_right = len(right_labels)
number_total = len(y)

left_weight = number_left / number_total
right_weight = number_right / number_total
weighted_entropy = left_weight * left_entropy + right_weight * right_entropy

parent_counts = np.array([np.sum(y == 0), np.sum(y == 1)])
parent_proportions = parent_counts / parent_counts.sum()
parent_entropy = entropy(parent_proportions, base=2)

print('Left weight:', left_weight)
print('Right weight:', right_weight)
print('Parent entropy:', round(parent_entropy, 3))
print('Weighted entropy after x2 < 3:', round(weighted_entropy, 3))
print('Entropy reduction:', round(parent_entropy - weighted_entropy, 3))

## 9. Calculate the score for every `x1` split

We have calculated one split by hand. Now let's repeat the same calculation for every candidate threshold in `x1`.

The loop below is deliberately explicit. For each threshold, we make the left and right masks, count both classes, turn counts into proportions, calculate each entropy, and then calculate the weighted average. We store the threshold and its score so we can compare them afterwards.

In [ ]:
x1_split_results = []

for threshold in split_x1:
    # Separate the observations using this x1 threshold.
    left_mask = X[:, 0] < threshold
    right_mask = ~left_mask
    left_labels = y[left_mask]
    right_labels = y[right_mask]

    # Count class 0 and class 1 on the left.
    left_counts = np.array([
        np.sum(left_labels == 0),
        np.sum(left_labels == 1)
    ])
    left_proportions = left_counts / left_counts.sum()
    left_entropy = entropy(left_proportions[left_proportions > 0], base=2)

    # Count class 0 and class 1 on the right.
    right_counts = np.array([
        np.sum(right_labels == 0),
        np.sum(right_labels == 1)
    ])
    right_proportions = right_counts / right_counts.sum()
    right_entropy = entropy(right_proportions[right_proportions > 0], base=2)

    # Give each child entropy a weight based on its number of observations.
    score = (
        len(left_labels) / len(y) * left_entropy
        + len(right_labels) / len(y) * right_entropy
    )
    x1_split_results.append([threshold, score])

print('x1 threshold and weighted entropy:')
for threshold, score in x1_split_results:
    print('x1 <', threshold, 'score =', round(score, 3))

## 10. Calculate the score for every `x2` split

Now we repeat the calculation using `x2`. The code is almost the same; the main difference is that the split mask uses column 1 instead of column 0.

Keeping this as a separate cell lets us compare the `x1` and `x2` splits and see that a tree checks each feature.

In [ ]:
x2_split_results = []

for threshold in split_x2:
    # Separate the observations using this x2 threshold.
    left_mask = X[:, 1] < threshold
    right_mask = ~left_mask
    left_labels = y[left_mask]
    right_labels = y[right_mask]

    # Count the classes on the left, then calculate its entropy.
    left_counts = np.array([
        np.sum(left_labels == 0),
        np.sum(left_labels == 1)
    ])
    left_proportions = left_counts / left_counts.sum()
    left_entropy = entropy(left_proportions[left_proportions > 0], base=2)

    # Count the classes on the right, then calculate its entropy.
    right_counts = np.array([
        np.sum(right_labels == 0),
        np.sum(right_labels == 1)
    ])
    right_proportions = right_counts / right_counts.sum()
    right_entropy = entropy(right_proportions[right_proportions > 0], base=2)

    # Combine the two entropies using the child-group sizes as weights.
    score = (
        len(left_labels) / len(y) * left_entropy
        + len(right_labels) / len(y) * right_entropy
    )
    x2_split_results.append([threshold, score])

print('x2 threshold and weighted entropy:')
for threshold, score in x2_split_results:
    print('x2 <', threshold, 'score =', round(score, 3))

## 11. Compare all the first splits

A lower weighted entropy means that the two child groups are less mixed. Let's put the results for both features into one table and sort from the lowest score to the highest.

The best split for this dataset should be the rule `x2 < 3`. Check that the first row of the table agrees with the result we calculated by hand.

In [ ]:
import pandas as pd

# Add the feature name to each result so we can read the table.
all_split_results = []

for threshold, score in x1_split_results:
    all_split_results.append(['x1', threshold, score])

for threshold, score in x2_split_results:
    all_split_results.append(['x2', threshold, score])

split_table = pd.DataFrame(
    all_split_results,
    columns=['feature', 'threshold', 'weighted entropy']
)
split_table = split_table.sort_values('weighted entropy')
display(split_table.round(3))

### Stop and check the split scores

1. Which feature and threshold have the lowest weighted entropy?
2. Why is the best score lower than the parent entropy?
3. Do any other splits have a similar score? What does that tell you about the data?

## 12. Make the first split in the plot

The best rule is `x2 < 3`. This is a horizontal line because `x2` is on the vertical axis. Points below the line go to the left child in our tree; points above or on the line go to the right child.

Let's draw the line and check which observations went to each side.

In [ ]:
first_feature = 'x2'
first_threshold = 3

plt.scatter(X[:, 0], X[:, 1], c=colors[y], s=90, edgecolor='black')
for row_number in range(len(X)):
    plt.annotate(
        str(row_number),
        (X[row_number, 0], X[row_number, 1]),
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.axhline(first_threshold, color='black', linestyle='--', label='x2 < 3')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('The first decision-tree split')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

left_mask = X[:, 1] < first_threshold
print('Labels below the line:', y[left_mask])
print('Labels at or above the line:', y[~left_mask])

The lower group contains only class 0, so the tree can make that a leaf. The upper group still contains both classes. A decision tree now repeats the split search **inside that group only**. This is the next step in growing the tree.

## 13. Keep the branch that still needs splitting

The branch that remains mixed is `x2 >= 3`. We make a mask for those observations and use it to select the feature rows and labels. The other branch is already pure, so we do not need to split it again.

In [ ]:
right_branch_mask = X[:, 1] >= first_threshold
X_right_branch = X[right_branch_mask]
y_right_branch = y[right_branch_mask]

print('Observations in the branch x2 >= 3:')
for row_number in range(len(X_right_branch)):
    print(
        'x1 =', X_right_branch[row_number, 0],
        ', x2 =', X_right_branch[row_number, 1],
        ', class =', y_right_branch[row_number]
    )
print('Class counts in this branch:', np.bincount(y_right_branch, minlength=2))

## 14. Find possible splits inside this branch

The branch has seven observations. We must find candidate thresholds using only these seven observations, not the whole dataset. This is how a tree builds a split below an earlier split.

Let's calculate the midpoint thresholds for `x1` and `x2` in this branch and print them.

In [ ]:
right_x1_values = np.unique(X_right_branch[:, 0])
right_split_x1 = (right_x1_values[1:] + right_x1_values[:-1]) / 2
print('Possible x1 splits in this branch:', right_split_x1)

right_x2_values = np.unique(X_right_branch[:, 1])
right_split_x2 = (right_x2_values[1:] + right_x2_values[:-1]) / 2
print('Possible x2 splits in this branch:', right_split_x2)

## 15. Try the `x1 < 5` split in this branch

The candidate `x1 < 5` separates the remaining observations into two smaller groups. Let's print those groups first, before calculating the entropy.

In [ ]:
second_threshold = 5

left_of_second_split = X_right_branch[:, 0] < second_threshold
right_of_second_split = ~left_of_second_split

print('Labels where x1 < 5:', y_right_branch[left_of_second_split])
print('Labels where x1 >= 5:', y_right_branch[right_of_second_split])
print('Counts where x1 < 5:', np.bincount(y_right_branch[left_of_second_split], minlength=2))
print('Counts where x1 >= 5:', np.bincount(y_right_branch[right_of_second_split], minlength=2))

The right group is now pure class 1. The left group still contains two class-0 observations and one class-1 observation. It is smaller than the original branch, but it is still mixed, so the tree has more work to do.

## 16. Score the `x1 < 5` split

We will calculate the entropy and weighted score again, this time using the seven observations in the current branch as the parent group. Do not use `len(y)` here; the weights must be based on the size of this branch.

In [ ]:
left_labels = y_right_branch[left_of_second_split]
right_labels = y_right_branch[right_of_second_split]

left_counts = np.bincount(left_labels, minlength=2)
right_counts = np.bincount(right_labels, minlength=2)

left_proportions = left_counts / left_counts.sum()
right_proportions = right_counts / right_counts.sum()

left_entropy = entropy(left_proportions[left_proportions > 0], base=2)
right_entropy = entropy(right_proportions[right_proportions > 0], base=2)

branch_size = len(y_right_branch)
weighted_branch_entropy = (
    len(left_labels) / branch_size * left_entropy
    + len(right_labels) / branch_size * right_entropy
)

print('Left entropy:', round(left_entropy, 3))
print('Right entropy:', round(right_entropy, 3))
print('Weighted entropy within this branch:', round(weighted_branch_entropy, 3))

## 17. Draw the second split

The second rule is `x1 < 5`, but it only applies to observations that already passed the first rule `x2 >= 3`. That is why the line starts at `x2 = 3` rather than crossing the whole plot.

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=colors[y], s=90, edgecolor='black')
for row_number in range(len(X)):
    plt.annotate(
        str(row_number),
        (X[row_number, 0], X[row_number, 1]),
        xytext=(5, 5),
        textcoords='offset points'
    )

# First split: below x2=3 goes to the class-0 leaf.
plt.axhline(3, color='green', linestyle='--', label='first split: x2 < 3')

# Second split applies only above the first line.
plt.plot([5, 5], [3, 10], color='blue', linestyle='--', label='second split: x1 < 5')

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Two levels of the decision tree')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 18. Split the remaining mixed group

The mixed group has three observations: two from class 0 and one from class 1. Their `x2` values are 4, 6 and 8. A threshold halfway between 6 and 8 will separate the two class-0 observations from the class-1 observation.

Let's print the three rows and calculate that midpoint.

In [ ]:
mixed_branch_X = X_right_branch[left_of_second_split]
mixed_branch_y = y_right_branch[left_of_second_split]

print('Features in the mixed branch:')
print(mixed_branch_X)
print('Labels in the mixed branch:', mixed_branch_y)

third_threshold = (6 + 8) / 2
print('A split between x2=6 and x2=8 is:', third_threshold)

The threshold is `x2 < 7`. The two observations below the threshold are both class 0, and the observation above the threshold is class 1. These groups are pure, so they can become leaves.

In [ ]:
third_left_mask = mixed_branch_X[:, 1] < third_threshold
third_right_mask = ~third_left_mask

print('Class labels where x2 < 7:', mixed_branch_y[third_left_mask])
print('Class labels where x2 >= 7:', mixed_branch_y[third_right_mask])

## Draw the third split

The third rule only applies inside the branch where `x2 >= 3` and `x1 < 5`. We draw it as a short horizontal line, to show that the tree does not apply this rule to every point in the plot.


In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=colors[y], s=90, edgecolor='black')
for row_number in range(len(X)):
    plt.annotate(
        str(row_number),
        (X[row_number, 0], X[row_number, 1]),
        xytext=(5, 5),
        textcoords='offset points'
    )

# The first split is horizontal across the full plot.
plt.axhline(3, color='green', linestyle='--', label='first: x2 < 3')

# The second split is only used above the first split.
plt.plot([5, 5], [3, 10], color='blue', linestyle='--', label='second: x1 < 5')

# The third split only applies left of x1=5 and above x2=3.
plt.plot([0, 5], [7, 7], color='purple', linestyle='--', label='third: x2 < 7')

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Three split rules in the decision tree')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 20. Put the rules together

We have now made three splits:

1. If `x2 < 3`, predict class 0.
2. Otherwise, if `x1 >= 5`, predict class 1.
3. Otherwise, use `x2 < 7` to choose between class 0 and class 1.

Let's write these questions in order for one observation. The order matters: each later question is only checked if the earlier question sends the observation down that branch.

In [ ]:
def predict_with_tree_rules(point):
    x1_value = point[0]
    x2_value = point[1]

    # First question at the root of the tree.
    if x2_value < 3:
        return 0

    # This question applies only when x2 was at least 3.
    if x1_value >= 5:
        return 1

    # This question applies only when x1 was below 5.
    if x2_value < 7:
        return 0
    else:
        return 1

# Try the rules on the first observation.
print('First observation:', X[0])
print('Predicted class:', predict_with_tree_rules(X[0]))
print('Actual class:', y[0])

Now let's apply the same rules to every observation. We will compare each prediction with the known class label. The tree can classify this tiny dataset perfectly, but that does not mean it will work perfectly on new data. We built the rules by looking at these ten observations.

In [ ]:
tree_predictions = []

for row_number in range(len(X)):
    prediction = predict_with_tree_rules(X[row_number])
    tree_predictions.append(prediction)
    print(
        'Observation', row_number,
        '| true class =', y[row_number],
        '| predicted class =', prediction
    )

tree_predictions = np.array(tree_predictions)
number_correct = np.sum(tree_predictions == y)
print('Number correct:', number_correct, 'out of', len(y))

## 20. What did we do?

We did not ask a library to build the tree for us. We made the data visible, tried a candidate split, counted the classes, calculated entropy, searched other thresholds, and repeated the search inside a mixed branch.

A library can do these calculations much faster and can grow much larger trees. But the basic idea is the same: ask a feature question, measure the class mixing on each side, and continue splitting branches that are still mixed.

### Check your understanding

1. Why do we use midpoints between neighboring feature values as candidate thresholds?
2. Why is the entropy of the left branch weighted by its number of observations?
3. Why did we calculate new split points using only the `x2 >= 3` branch?
4. Which questions does the final tree ask, and in what order?
5. Why does perfect classification of these ten points not show that the tree will work on new data?

Next, we will work through KNN in a similar way: calculate distances, sort them, and let the nearest observations vote.